In [4]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler, LabelEncoder
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.base import BaseEstimator, ClassifierMixin

from ga_preprocessing import ga_feature_selection

# Load dataset
df = pd.read_csv('learning_disability_dataset.csv')

# Encode categorical features
df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})
df['Sleep Quality'] = df['Sleep Quality'].map({'Poor': 0, 'Average': 1, 'Good': 2})
df['Family History'] = df['Family History'].map({'No': 0, 'Yes': 1})

# Features & Labels
X = df.drop(columns=['Labels'])
y_raw = df['Labels'].apply(
    lambda x: [
        label.strip().capitalize()
        for label in str(x).split(',')
        if label.strip().lower() not in ['nan', '', 'none']
    ]
)

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(y_raw)

# Prepare single-label format for GA
y_single = np.array([x[0] if x else 'None' for x in y_raw.tolist()])
le = LabelEncoder()
y_encoded = le.fit_transform(y_single)

# GA Feature Selection
print("🔍 Running GA-based Feature Selection...")
X_np = X.values
selected_indices = ga_feature_selection(X_np, y_encoded, ngen=30, pop_size=40)
X = X.iloc[:, selected_indices]

# Save selected feature names
selected_feature_names = X.columns.tolist()
os.makedirs('saved_models', exist_ok=True)
joblib.dump(selected_feature_names, 'saved_models/selected_features.pkl')
print("🧬 Final features used in training:", selected_feature_names)

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
joblib.dump(scaler, 'saved_models/scaler.pkl')

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Deep learning model
def create_nn_model():
    model = Sequential()
    model.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
    model.add(Dropout(0.3))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(y_train.shape[1], activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Custom Keras wrapper
class KerasMultiOutputClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, build_fn, epochs=50, batch_size=16, verbose=0):
        self.build_fn = build_fn
        self.epochs = epochs
        self.batch_size = batch_size
        self.verbose = verbose

    def fit(self, X, y):
        self.model = self.build_fn()
        self.model.fit(
            X, y,
            epochs=self.epochs,
            batch_size=self.batch_size,
            verbose=self.verbose,
            callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
        )
        return self

    def predict(self, X):
        pred = self.model.predict(X)
        return (pred > 0.5).astype(int)

# Define all models
models = {
    'random_forest': MultiOutputClassifier(RandomForestClassifier(n_estimators=100, random_state=42)),
    'svm': MultiOutputClassifier(SVC(probability=True, kernel='linear', random_state=42)),
    'logistic_regression': MultiOutputClassifier(LogisticRegression(max_iter=1000, random_state=42)),
    'knn': MultiOutputClassifier(KNeighborsClassifier(n_neighbors=5)),
    'gradient_boosting': MultiOutputClassifier(GradientBoostingClassifier(n_estimators=100, random_state=42)),
    'deep_learning': KerasMultiOutputClassifier(create_nn_model, epochs=30, batch_size=16, verbose=1)
}

# Train and evaluate
for name, model in models.items():
    print(f"\n🔹 Training {name} model...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred = np.nan_to_num(y_pred, nan=0.0).astype(int)

    print(f"📊 Classification Report for {name}:")
    try:
        print(classification_report(
            y_test,
            y_pred,
            target_names=mlb.classes_,
            labels=range(len(mlb.classes_))
        ))
    except ValueError as e:
        print("⚠️ Report issue:", str(e))

    if name != "deep_learning":
        joblib.dump(model, f'saved_models/{name}_model.pkl')
    else:
        model.model.save("saved_models/deep_learning_model.h5")

joblib.dump(mlb, 'saved_models/mlb.pkl')
print("\n✅ All models trained and saved.")




🔍 Running GA-based Feature Selection...


c:\Users\towqe\AppData\Local\Programs\Python\Python311\Lib\site-packages\deap\creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
c:\Users\towqe\AppData\Local\Programs\Python\Python311\Lib\site-packages\deap\creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


gen	nevals
0  	40    
1  	22    
2  	26    
3  	19    
4  	20    
5  	23    
6  	26    
7  	25    
8  	25    
9  	24    
10 	22    
11 	19    
12 	24    
13 	26    
14 	24    
15 	30    
16 	29    
17 	23    
18 	23    
19 	19    
20 	19    
21 	18    
22 	25    
23 	22    
24 	18    
25 	26    
26 	24    
27 	26    
28 	27    
29 	26    
30 	10    
🧬 Final features used in training: ['Age', 'Sleep Quality', 'Attention Span', 'Memory Retention', 'Visual Processing', 'Reading Score', 'Verbal Reasoning', 'Spelling Accuracy']

🔹 Training random_forest model...
📊 Classification Report for random_forest:
              precision    recall  f1-score   support

        Adhd       0.89      0.80      0.84        70
    Dyslexia       0.94      0.85      0.89        97

   micro avg       0.92      0.83      0.87       167
   macro avg       0.92      0.82      0.87       167
weighted avg       0.92      0.83      0.87       167
 samples avg       0.56      0.52      0.54       167



c:\Users\towqe\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\towqe\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\towqe\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(aver


🔹 Training svm model...
📊 Classification Report for svm:


...

Epoch 1/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.5124 - loss: 0.6695
Epoch 2/30
26/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4877 - loss: 0.5630

c:\Users\towqe\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\callbacks\early_stopping.py:153: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)


50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4910 - loss: 0.5455
Epoch 3/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5881 - loss: 0.4489
Epoch 4/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6443 - loss: 0.3836
Epoch 5/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6261 - loss: 0.3693
Epoch 6/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6224 - loss: 0.3479
Epoch 7/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6271 - loss: 0.3569
Epoch 8/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6684 - loss: 0.3161
Epoch 9/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6274 - loss: 0.3110
Epoch 10/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6388 - loss: 0.3288
Epoch 11/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6482 - loss: 0.3352
Epoch 12/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6354 - loss: 0.3195
Epoch 13/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6380 - loss: 0.3283

c:\Users\towqe\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\towqe\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\towqe\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(aver

📊 Classification Report for deep_learning:
              precision    recall  f1-score   support

        Adhd       0.83      0.81      0.82        70
    Dyslexia       0.93      0.84      0.88        97

   micro avg       0.88      0.83      0.85       167
   macro avg       0.88      0.82      0.85       167
weighted avg       0.89      0.83      0.86       167
 samples avg       0.57      0.53      0.54       167


✅ All models trained and saved.
